Universal Setup 

In [17]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

results = {"Logistic Regression": (acc, auc)}

# Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            numTrees=100, maxDepth=10, seed=42)
rf_model = rf.fit(train)
rf_preds = rf_model.transform(test)
rf_acc = MulticlassClassificationEvaluator(
    labelCol="label", metricName="accuracy").evaluate(rf_preds)
rf_auc = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC").evaluate(rf_preds)
results["Random Forest"] = (rf_acc, rf_auc)

# Gradient-Boosted Trees
gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    maxIter=50, seed=42)
gbt_model = gbt.fit(train)
gbt_preds = gbt_model.transform(test)
gbt_acc = MulticlassClassificationEvaluator(
    labelCol="label", metricName="accuracy").evaluate(gbt_preds)
gbt_auc = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC").evaluate(gbt_preds)
results["Gradient-Boosted Trees"] = (gbt_acc, gbt_auc)

print("\n" + "="*50)
print(f"{'Model':<25}{'Accuracy':<12}{'AUC':<10}")
print("="*50)
for name, (a, u) in results.items():
    print(f"{name:<25}{a:<12.4f}{u:<10.4f}")
print("="*50)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("Day7Project") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")


Model                    Accuracy    AUC       
Logistic Regression      0.8261      0.8731    
Random Forest            0.8043      0.8942    
Gradient-Boosted Trees   0.7826      0.8615    
Spark version: 4.2.0


#Load the Data

In [7]:
columns = ["age","sex","cp","trestbps","chol","fbs","restecg",
           "thalach","exang","oldpeak","slope","ca","thal","target"]

df =df = spark.read.csv("day7_data/heart_disease.csv",
                    header=False,
                    inferSchema=True,
                    nanValue="?")
for i, name in enumerate(columns):
    df = df.withColumnRenamed(f"_c{i}", name)

print("Rows:", df.count())
df.show(5)
df.groupBy("target").count().orderBy("target").show()

Rows: 303
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
| age|sex| cp|trestbps| chol|fbs|restecg|thalach|exang|oldpeak|slope| ca|thal|target|
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
|63.0|1.0|1.0|   145.0|233.0|1.0|    2.0|  150.0|  0.0|    2.3|  3.0|0.0| 6.0|     0|
|67.0|1.0|4.0|   160.0|286.0|0.0|    2.0|  108.0|  1.0|    1.5|  2.0|3.0| 3.0|     2|
|67.0|1.0|4.0|   120.0|229.0|0.0|    2.0|  129.0|  1.0|    2.6|  2.0|2.0| 7.0|     1|
|37.0|1.0|3.0|   130.0|250.0|0.0|    0.0|  187.0|  0.0|    3.5|  3.0|0.0| 3.0|     0|
|41.0|0.0|2.0|   130.0|204.0|0.0|    2.0|  172.0|  0.0|    1.4|  1.0|0.0| 3.0|     0|
+----+---+---+--------+-----+---+-------+-------+-----+-------+-----+---+----+------+
only showing top 5 rows
+------+-----+
|target|count|
+------+-----+
|     0|  164|
|     1|   55|
|     2|   36|
|     3|   35|
|     4|   13|
+------+-----+



 Clean & Binarize Label

In [8]:
clean = df.dropna()
print("Rows after dropna:", clean.count())

clean = clean.withColumn("target", col("target").cast(DoubleType()))
clean = clean.withColumn("label", when(col("target") > 0, 1.0).otherwise(0.0))
clean = clean.drop("target")

clean.groupBy("label").count().orderBy("label").show()

Rows after dropna: 297
+-----+-----+
|label|count|
+-----+-----+
|  0.0|  160|
|  1.0|  137|
+-----+-----+



 Features & Scaling

In [9]:
feature_cols = ["age","sex","cp","trestbps","chol","fbs","restecg",
                "thalach","exang","oldpeak","slope","ca","thal"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
assembled = assembler.transform(clean)

scaler = StandardScaler(inputCol="raw_features", outputCol="features",
                        withMean=True, withStd=True)
final = scaler.fit(assembled).transform(assembled).select("features","label")
final.show(3, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                                                  |label|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|[0.9346032586984205,0.6899303003111266,-2.236853510480693,0.7491157101964772,-0.2759776060833338,2.4263321977007934,1.0084964074436102,0.017464957251029093,-0.6952460020503559,1.067164164714716,2.260330479291401,-0.7207595

Train/Test Split & Logistic Regression

In [10]:
train, test = final.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count()}, Test: {test.count()}")

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=100)
model = lr.fit(train)
predictions = model.transform(test)
predictions.select("label","prediction","probability").show(5, truncate=False)

Train: 251, Test: 46
+-----+----------+-----------------------------------------+
|label|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0.0  |0.0       |[0.9603242757319422,0.039675724268057766]|
|1.0  |1.0       |[0.15272355607325663,0.8472764439267434] |
|0.0  |0.0       |[0.6322416302169585,0.36775836978304155] |
|1.0  |1.0       |[0.32452530581677647,0.6754746941832235] |
|0.0  |0.0       |[0.9741022689310296,0.02589773106897042] |
+-----+----------+-----------------------------------------+
only showing top 5 rows


EVALUATE

In [11]:
acc = MulticlassClassificationEvaluator(
    labelCol="label", metricName="accuracy").evaluate(predictions)
auc = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC").evaluate(predictions)

print(f"Logistic Regression — Accuracy: {acc:.4f}, AUC: {auc:.4f}")
print(f"Required threshold:     Accuracy > 0.70, AUC > 0.70")

predictions.groupBy("label","prediction").count() \
    .orderBy("label","prediction").show()

Logistic Regression — Accuracy: 0.8261, AUC: 0.8731
Required threshold:     Accuracy > 0.70, AUC > 0.70
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|   21|
|  0.0|       1.0|    5|
|  1.0|       0.0|    3|
|  1.0|       1.0|   17|
+-----+----------+-----+



Feature Importance

In [12]:
importance = list(zip(feature_cols, model.coefficients.toArray()))
importance.sort(key=lambda x: abs(x[1]), reverse=True)

print("Top 5 most influential features:")
for name, coef in importance[:5]:
    direction = "↑ risk" if coef > 0 else "↓ risk"
    print(f"  {name:10s}  coef={coef:+.4f}  ({direction})")

Top 5 most influential features:
  ca          coef=+1.3920  (↑ risk)
  thal        coef=+0.7411  (↑ risk)
  sex         coef=+0.6496  (↑ risk)
  trestbps    coef=+0.5642  (↑ risk)
  thalach     coef=-0.4941  (↓ risk)


 Compare with Random Forest & GBT

In [16]:
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier

results = {"Logistic Regression": (acc, auc)}

# Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            numTrees=100, maxDepth=10, seed=42)
rf_model = rf.fit(train)
rf_preds = rf_model.transform(test)
rf_acc = MulticlassClassificationEvaluator(
    labelCol="label", metricName="accuracy").evaluate(rf_preds)
rf_auc = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC").evaluate(rf_preds)
results["Random Forest"] = (rf_acc, rf_auc)

# Gradient-Boosted Trees
gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    maxIter=50, seed=42)
gbt_model = gbt.fit(train)
gbt_preds = gbt_model.transform(test)
gbt_acc = MulticlassClassificationEvaluator(
    labelCol="label", metricName="accuracy").evaluate(gbt_preds)
gbt_auc = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC").evaluate(gbt_preds)
results["Gradient-Boosted Trees"] = (gbt_acc, gbt_auc)

print("\n" + "="*50)
print(f"{'Model':<25}{'Accuracy':<12}{'AUC':<10}")
print("="*50)
for name, (a, u) in results.items():
    print(f"{name:<25}{a:<12.4f}{u:<10.4f}")
print("="*50)


Model                    Accuracy    AUC       
Logistic Regression      0.8261      0.8731    
Random Forest            0.8043      0.8942    
Gradient-Boosted Trees   0.7826      0.8615    


In [18]:
print("="*50)
print("Random Forest — Confusion Matrix (Test Set)")
print("="*50)

cm = rf_preds.groupBy("label","prediction").count().orderBy("label","prediction").collect()
matrix = {}
for row in cm:
    matrix[(row["label"], row["prediction"])] = row["count"]

tn = matrix.get((0.0, 0.0), 0)
fp = matrix.get((0.0, 1.0), 0)
fn = matrix.get((1.0, 0.0), 0)
tp = matrix.get((1.0, 1.0), 0)

print(f"              Predicted No    Predicted Yes")
print(f"Actual No    {tn:>10}    {fp:>10}")
print(f"Actual Yes   {fn:>10}    {tp:>10}")
print()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")
print(f"\nPrecision (disease): {precision:.4f}")
print(f"Recall    (disease): {recall:.4f}")
print(f"F1 Score            : {f1:.4f}")

Random Forest — Confusion Matrix (Test Set)
              Predicted No    Predicted Yes
Actual No            23             3
Actual Yes            6            14

True Negatives:  23
False Positives: 3
False Negatives: 6
True Positives:  14

Precision (disease): 0.8235
Recall    (disease): 0.7000
F1 Score            : 0.7568
